In [1]:
import requests
import pandas as pd
from datetime import datetime
import time

In [2]:
links = [
    "https://www.reddit.com/r/phcareers/comments/17r76bz/ive_been_job_searching_for_more_than_a_month_now/",
    "https://www.reddit.com/r/phcareers/comments/1b7a0m2/i_never_knew_looking_for_work_would_be_this_hard/",
    "https://www.reddit.com/r/phcareers/comments/1iy2bg1/mahirap_pala_ang_job_hunting_especially_if_from/",
    "https://www.reddit.com/r/phcareers/comments/zfs15j/resigning_even_without_a_new_job/",
    "https://www.reddit.com/r/phcareers/comments/14rylp9/ayoko_ko_na_magtrabaho_sa_government_natin/",
    "https://www.reddit.com/r/phcareers/comments/1d7wjm4/my_goal_was_to_have_a_new_job_by_june_this_june_i/",
    "https://www.reddit.com/r/phcareers/comments/1i3x4m1/how_is_our_job_hunting_early_this_2025_so_far/",
    "https://www.reddit.com/r/phcareers/comments/1jsxgck/random_help_thread_april_07_to_april_13_2025/",
    "https://www.reddit.com/r/phcareers/comments/190vcqj/random_help_thread_january_08_to_january_14_2024/",
    "https://www.reddit.com/r/phcareers/comments/1j7d686/the_recruiter_who_ghosted_me_suddenly_came_back/",
    "https://www.reddit.com/r/phcareers/comments/171254h/i_was_ghosted_after_having_been_scheduled_for_a/",
    "https://www.reddit.com/r/phcareers/comments/15rd83t/ghosted_by_an_employer_who_promised_me_sht_what/",
    "https://www.reddit.com/r/phcareers/comments/1hxwi6g/thoughts_on_ghostingbeing_ghosted_during_job/",
    "https://www.reddit.com/r/phcareers/comments/16tt3y5/why_is_my_jo_taking_so_long_am_i_ghosted_by_the/",
    "https://www.reddit.com/r/phcareers/comments/16zcg4k/how_to_deal_with_ghosting_recruiters/",
    "https://www.reddit.com/r/phcareers/comments/1k7eafo/why_do_some_companies_ignore_applicants_after_the/",
    "https://www.reddit.com/r/phcareers/comments/1imwgne/i_was_ghosted_by_hr_after_they_told_me_i_got_the/",
    "https://www.reddit.com/r/phcareers/comments/1490t09/109_applications_in_and_still_no_luck_landing_a/",
    "https://www.reddit.com/r/phcareers/comments/17oxhgu/unknown_number_called_me_offering_an_online_job/",
    "https://www.reddit.com/r/phcareers/comments/114hij0/should_you_be_emailed_back_by_recruiters_to_tell/",
    "https://www.reddit.com/r/phcareers/comments/16n0k3r/sr_recruitment_manager_here_to_answer_your/",
    "https://www.reddit.com/r/phcareers/comments/12b1fja/unemployed_lost_and_losing_hope/",
    "https://www.reddit.com/r/phcareers/comments/116wamy/hundreds_of_applications_dozens_of_rejections/"
]

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

def assign_theme(text):
    text = text.lower()
    if "ghost" in text or "no reply" in text or "no response" in text:
        return "GHOSTING"
    elif "hard" in text or "difficult" in text or "draining" in text:
        return "EMOTIONAL"
    elif "scam" in text:
        return "SCAM"
    elif "resume" in text:
        return "VISIBILITY"
    elif "application" in text:
        return "PROCESS"
    else:
        return "OTHER"

rows = []

for url in links:
    json_url = url.rstrip("/") + ".json?raw_json=1"

    success = False

    for attempt in range(3):
        try:
            res = requests.get(json_url, headers=headers)

            print(res.status_code, url)

            if res.status_code == 200 and res.text.strip():
                data = res.json()
                success = True
                break
            else:
                time.sleep(3)

        except:
            time.sleep(3)

    if not success:
        print(f"Skipped: {url}")
        continue

    # -------------------------
    # POST
    # -------------------------
    try:
        post = data[0]["data"]["children"][0]["data"]
    except:
        continue

    post_content = post["selftext"].strip() if post["selftext"] else post["title"]

    rows.append({
        "Source Subreddit": post.get("subreddit", ""),
        "Post URL": url,
        "Post Title": post.get("title", ""),
        "Post or Comment": "post",
        "Content (Full Text)": post_content,
        "Date Posted": datetime.fromtimestamp(post["created_utc"]).strftime("%m/%Y"),
        "Upvotes / Score": post.get("score", 0),
        "Theme Tag": assign_theme(post_content),
        "Platform Mentioned": "None",
        "Philippine Context?": "Yes" if post.get("subreddit","").lower().startswith("ph") else "No",
        "Notable Quote": post_content[:200],
        "Intern Notes": ""
    })

    # -------------------------
    # COMMENTS
    # -------------------------
    try:
        comments = data[1]["data"]["children"]
    except:
        comments = []

    for comment in comments:
        if comment.get("kind") != "t1":
            continue

        c = comment["data"]
        body = c.get("body", "").strip()

        # skip deleted
        if not body or body in ["[deleted]", "[removed]"]:
            continue

        rows.append({
            "Source Subreddit": post.get("subreddit", ""),
            "Post URL": url,
            "Post Title": post.get("title", ""),
            "Post or Comment": "comment",
            "Content (Full Text)": body,
            "Date Posted": datetime.fromtimestamp(c["created_utc"]).strftime("%m/%Y"),
            "Upvotes / Score": c.get("score", 0),
            "Theme Tag": assign_theme(body),
            "Platform Mentioned": "None",
            "Philippine Context?": "Yes" if post.get("subreddit","").lower().startswith("ph") else "No",
            "Notable Quote": body[:200],
            "Intern Notes": ""
        })

    time.sleep(4)

df = pd.DataFrame(rows)
df.to_csv("reddit_data.csv", index=False, encoding="utf-8-sig")

print("Done. Rows collected:", len(df))

200 https://www.reddit.com/r/phcareers/comments/17r76bz/ive_been_job_searching_for_more_than_a_month_now/
200 https://www.reddit.com/r/phcareers/comments/1b7a0m2/i_never_knew_looking_for_work_would_be_this_hard/
200 https://www.reddit.com/r/phcareers/comments/1iy2bg1/mahirap_pala_ang_job_hunting_especially_if_from/
200 https://www.reddit.com/r/phcareers/comments/zfs15j/resigning_even_without_a_new_job/
200 https://www.reddit.com/r/phcareers/comments/14rylp9/ayoko_ko_na_magtrabaho_sa_government_natin/
200 https://www.reddit.com/r/phcareers/comments/1d7wjm4/my_goal_was_to_have_a_new_job_by_june_this_june_i/
200 https://www.reddit.com/r/phcareers/comments/1i3x4m1/how_is_our_job_hunting_early_this_2025_so_far/
200 https://www.reddit.com/r/phcareers/comments/1jsxgck/random_help_thread_april_07_to_april_13_2025/
200 https://www.reddit.com/r/phcareers/comments/190vcqj/random_help_thread_january_08_to_january_14_2024/
200 https://www.reddit.com/r/phcareers/comments/1j7d686/the_recruiter_who_gh